# 04 — Swap candidates: which Latinate words can we replace?

**Highlights**
- The swap dictionary (fancy Latin-root word → plain Germanic word) is built from the corpus's *Latinate* word types. This notebook just lists them so we can eyeball the candidates before building anything.
- Reality check up front: with the high-confidence per-word labeller, only **~700 word types** in the whole 10M corpus get a solid *Latinate* label. They live in the **rare tail** — the 500th most common one sits around global rank ~47,000.
- **§1** lists the 500 most common Latinate words (plus what fraction of the corpus they actually cover). **§2** bins them with notebook 02's exact frequency bins and lists the top 100 per bin.

**Why this matters:** the rarer these words are, the less text a swap can touch. A small, rare candidate set is the key constraint on how big the intervention can be.


## Setup

Reuses the shared labeller from `src/babylm_2026/etymology.py` (EtymoLink gold + Macro-Etym fallback, same as notebooks 02–03). We count word frequencies over the strict-small corpus, rank every type globally (function words included, to match notebook 02's bins), and label each one Germanic / Latinate / other.


In [1]:
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

# Locate the repo root and import the shared etymology labeller.
ROOT = Path.cwd()
ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from babylm_2026.etymology import corpus_word_frequencies, load_etymolink_gold, label_word

CORPUS = ROOT / "data" / "babylm_2026" / "strict_small"

freqs = corpus_word_frequencies(CORPUS)        # {word: count} over *.train.txt
ranked = freqs.most_common()                   # sorted high → low, defines global rank
total_tokens = sum(freqs.values())
gold = load_etymolink_gold()                   # high-confidence word → family

rows = []
for rank, (w, c) in enumerate(ranked):
    label, source = label_word(w, gold)
    rows.append((rank, w, c, label, source))

df = pd.DataFrame(rows, columns=["global_rank", "word", "freq", "etym_label", "etym_source"])
print(f"{len(df):,} word types  |  {total_tokens:,} tokens")
print()
print("Label distribution (every type in the corpus):")
print(df["etym_label"].value_counts())


/Users/deep/Uni/ss2026/llm-practical-course/projects/babylm-2026/.venv/lib/python3.13/site-packages/macroetym/main.py:27: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


110,990 word types  |  10,154,570 tokens

Label distribution (every type in the corpus):
etym_label
other       110109
Latinate       707
Germanic       174
Name: count, dtype: int64


## 1. The 500 most common Latinate words

These are our raw swap candidates, ranked by how common they are in the corpus. The `token_coverage` number is the punchline: it's the share of *all* word-tokens these 500 types account for. If it's small, swapping them changes only a sliver of the text.


In [2]:
lat = df[df["etym_label"] == "Latinate"].reset_index(drop=True)
top500 = lat.head(500)
coverage = top500["freq"].sum() / total_tokens

print(f"{len(lat)} Latinate types in total; showing the {len(top500)} most common.")
print(f"Global rank of the last one shown: {int(top500['global_rank'].iloc[-1]):,}")
print(f"Token coverage of these {len(top500)}: {coverage:.2%} of the whole corpus")
print()
print("Label source (confidence) for these 500:")
print(top500["etym_source"].value_counts())


707 Latinate types in total; showing the 500 most common.
Global rank of the last one shown: 46,767
Token coverage of these 500: 0.42% of the whole corpus

Label source (confidence) for these 500:
etym_source
etymolink    500
Name: count, dtype: int64


In [3]:
# Scannable grid: 50 rows x 10 cols, in descending-frequency order (left→right, top→bottom).
words = top500["word"].tolist()
words += [""] * (500 - len(words))
grid = pd.DataFrame(np.array(words, dtype=object).reshape(50, 10))
grid.index = [f"{i*10+1:>3}-{i*10+10:>3}" for i in range(50)]
grid.columns = [""] * 10
with pd.option_context("display.max_rows", 60, "display.max_colwidth", 20):
    display(grid)


,,,,,,,,,,
1- 10,car,family,river,november,london,hospital,aunt,chocolate,education,attention
11- 20,princess,period,various,event,role,governor,elizabeth,huge,jesus,female
21- 30,excellent,louis,jean,japan,emperor,federal,austria,asian,student,revolution
31- 40,operation,alice,admit,kansas,estate,virus,oliver,multiple,triangle,quality
41- 50,violence,universe,vision,basil,navy,agency,vast,irene,punishment,umbrella
51- 60,attend,notable,jolly,opera,mobile,urban,obvious,symbol,resistance,conscience
61- 70,julius,impression,aboard,indigenous,hebrew,generation,identify,gorgeous,entry,capitol
71- 80,application,magnificent,literary,violin,administrative,metro,tractor,versus,arizona,chorus
81- 90,journal,interfere,architecture,explosion,deny,attractive,formation,bucket,orthodox,facility
91-100,absurd,artistic,commissioner,examine,abortion,tobacco,immortal,qualification,manor,column


## 2. Binned by frequency (notebook 02's exact bins)

Same four bins as `02-confound-checks.ipynb` §2, by global rank in the pooled corpus: **top-1k / 1k–5k / 5k–25k / 25k+**. This shows *where* the Latinate words sit — if they all pile up in the rare bins, the common-word bins have almost nothing to swap.


In [4]:
BINS = [
    (0, 1000, "top-1k"),
    (1000, 5000, "1k–5k"),
    (5000, 25000, "5k–25k"),
    (25000, len(df), "25k+"),
]

def bin_of(rank):
    for lo, hi, name in BINS:
        if lo <= rank < hi:
            return name
    return None

lat["bin"] = lat["global_rank"].map(bin_of)

summary = []
for lo, hi, name in BINS:
    sub = lat[lat["bin"] == name]
    summary.append({
        "bin": name,
        "n_latinate_types": len(sub),
        "token_coverage": f"{sub['freq'].sum() / total_tokens:.3%}",
    })
display(pd.DataFrame(summary))


,bin,n_latinate_types,token_coverage
0,top-1k,6,0.116%
1,1k–5k,65,0.197%
2,5k–25k,265,0.097%
3,25k+,371,0.013%


In [5]:
# Top 100 most common Latinate words within each frequency bin.
for lo, hi, name in BINS:
    sub = lat[lat["bin"] == name].head(100)
    print(f"=== {name}: {len(sub)} Latinate (of {(lat['bin'] == name).sum()} in this bin) ===")
    print(", ".join(sub["word"].tolist()) if len(sub) else "(none)")
    print()


=== top-1k: 6 Latinate (of 6 in this bin) ===
car, family, river, november, london, hospital

=== 1k–5k: 65 Latinate (of 65 in this bin) ===
aunt, chocolate, education, attention, princess, period, various, event, role, governor, elizabeth, huge, jesus, female, excellent, louis, jean, japan, emperor, federal, austria, asian, student, revolution, operation, alice, admit, kansas, estate, virus, oliver, multiple, triangle, quality, violence, universe, vision, basil, navy, agency, vast, irene, punishment, umbrella, attend, notable, jolly, opera, mobile, urban, obvious, symbol, resistance, conscience, julius, impression, aboard, indigenous, hebrew, generation, identify, gorgeous, entry, capitol, application

=== 5k–25k: 100 Latinate (of 265 in this bin) ===
magnificent, literary, violin, administrative, metro, tractor, versus, arizona, chorus, journal, interfere, architecture, explosion, deny, attractive, formation, bucket, orthodox, facility, absurd, artistic, commissioner, examine, aborti

## Takeaway

- Confidently-Latinate vocabulary is **small and rare** in this corpus. That caps how much of the text a word-swap can touch — read the `token_coverage` figures as the ceiling on the intervention's reach.
- For the cheap first experiment, the top-N most common Latinate words are the right place to start: they give the most coverage per swap, and the list is small enough to hand-audit every replacement.
- Open question this raises for design: if even the top 500 cover only a few percent of tokens, is that enough signal to move a model — or do we need to widen the labeller (more "other" words are probably Latinate but uncovered by EtymoLink/Macro-Etym)?
